# Day 059 — Solution: Background Job Processing API

In [ ]:
_BACKGROUND_API_SRC = '"""background_api.py — Day 059: background job processing API.\n\nRun:  uvicorn background_api:app --reload\nDocs: http://localhost:8000/docs\n"""\nimport os\nimport threading\nimport uuid\nfrom datetime import datetime\n\nimport ollama\nfrom fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel, Field\n\nMODEL       = os.environ.get("MODEL", "llama3.2")\nAPP_VERSION = "1.0.0"\n\n\nclass _JobStore:\n    """Thread-safe in-memory job store."""\n\n    def __init__(self):\n        self._jobs: dict = {}\n        self._lock = threading.Lock()\n\n    def create(self, job_id: str) -> None:\n        with self._lock:\n            self._jobs[job_id] = {"status": "pending"}\n\n    def set_running(self, job_id: str) -> None:\n        with self._lock:\n            self._jobs[job_id] = {"status": "running"}\n\n    def set_done(self, job_id: str, result: str) -> None:\n        with self._lock:\n            self._jobs[job_id] = {"status": "done", "result": result}\n\n    def set_error(self, job_id: str, error: str) -> None:\n        with self._lock:\n            self._jobs[job_id] = {"status": "error", "error": error}\n\n    def get(self, job_id: str) -> dict | None:\n        with self._lock:\n            data = self._jobs.get(job_id)\n            return dict(data) if data else None\n\n\nclass SummarizeRequest(BaseModel):\n    text: str = Field(min_length=1)\n\n\ndef build_api(process_fn=None) -> FastAPI:\n    """Build the background job API.\n\n    process_fn: optional callable(text: str) -> str for testing.\n                If None, uses ollama.chat to summarize.\n    """\n    app = FastAPI(title="Background Job API", version=APP_VERSION)\n    store = _JobStore()\n\n    @app.get("/health")\n    def health():\n        return {\n            "status": "ok",\n            "timestamp": datetime.utcnow().isoformat() + "Z",\n            "version": APP_VERSION,\n        }\n\n    @app.post("/summarize", status_code=202)\n    def submit_summarize(req: SummarizeRequest):\n        job_id = uuid.uuid4().hex[:8]\n        store.create(job_id)\n\n        def worker():\n            store.set_running(job_id)\n            try:\n                if process_fn is not None:\n                    summary = process_fn(req.text)\n                else:\n                    prompt = "Summarize in one sentence:\\n\\n" + req.text[:3000]\n                    resp = ollama.chat(\n                        model=MODEL,\n                        messages=[{"role": "user", "content": prompt}],\n                    )\n                    summary = resp["message"]["content"]\n                store.set_done(job_id, summary)\n            except Exception as exc:\n                store.set_error(job_id, str(exc))\n\n        threading.Thread(target=worker, daemon=True).start()\n        return {"job_id": job_id, "status": "pending"}\n\n    @app.get("/jobs/{job_id}")\n    def get_job(job_id: str):\n        job = store.get(job_id)\n        if job is None:\n            raise HTTPException(status_code=404, detail="Job not found")\n        return job\n\n    return app\n\n\napp = build_api()\n\nif __name__ == "__main__":\n    import uvicorn\n    PORT = int(os.environ.get("PORT", "8000"))\n    uvicorn.run(app, host="0.0.0.0", port=PORT)\n'
from pathlib import Path
Path('background_api.py').write_text(_BACKGROUND_API_SRC)
print('background_api.py written.')

In [ ]:
import time
import threading
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

# ── inline test app (no Ollama required) ──────────────────────────────────────
import uuid

class _Store:
    def __init__(self):
        self._jobs: dict = {}
        self._lock = threading.Lock()
    def create(self, jid):
        with self._lock: self._jobs[jid] = {"status": "pending"}
    def done(self, jid, result):
        with self._lock: self._jobs[jid] = {"status": "done", "result": result}
    def error(self, jid, err):
        with self._lock: self._jobs[jid] = {"status": "error", "error": err}
    def get(self, jid):
        with self._lock: return dict(self._jobs.get(jid) or {}) or None

store = _Store()

class _SumReq(BaseModel):
    text: str = Field(min_length=1)

test_app = FastAPI()

@test_app.get("/health")
def _health():
    from datetime import datetime
    return {"status": "ok", "timestamp": datetime.utcnow().isoformat() + "Z",
            "version": "1.0.0"}

@test_app.post("/summarize", status_code=202)
def _submit(req: _SumReq):
    jid = uuid.uuid4().hex[:8]
    store.create(jid)
    def worker():
        time.sleep(0.01)
        store.done(jid, "Summary: " + req.text[:50])
    threading.Thread(target=worker, daemon=True).start()
    return {"job_id": jid, "status": "pending"}

@test_app.get("/jobs/{job_id}")
def _get(job_id: str):
    job = store.get(job_id)
    if not job: raise HTTPException(404, "Not found")
    return job

client = TestClient(test_app, raise_server_exceptions=False)

# /health
r = client.get("/health")
assert r.status_code == 200 and r.json()["status"] == "ok"
print("\u2705 /health works")

# POST /summarize → 202 + job_id
r2 = client.post("/summarize", json={"text": "This is a long document."})
assert r2.status_code == 202
body = r2.json()
assert "job_id" in body
print("\u2705 POST /summarize returns 202 + job_id")

# poll until done
jid = body["job_id"]
deadline = time.monotonic() + 2.0
while time.monotonic() < deadline:
    r3 = client.get(f"/jobs/{jid}")
    if r3.json()["status"] != "pending":
        break
    time.sleep(0.01)
assert r3.json()["status"] == "done"
assert "Summary:" in r3.json()["result"]
print("\u2705 GET /jobs/{id} returns result when done")

# 404 for unknown
r4 = client.get("/jobs/bad_id")
assert r4.status_code == 404
print("\u2705 unknown job_id \u2192 404")

# empty text → 422
r5 = client.post("/summarize", json={"text": ""})
assert r5.status_code == 422
print("\u2705 empty text \u2192 422")

print("\nDay 059 \u2014 Background Jobs & Queues complete! \U0001f389")
